# LOF with Post-Processing
Fills small gaps between detected points to catch edges of sustained anomalies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, confusion_matrix

## Load data and train LOF

In [ ]:
data = pd.read_csv('../data/labeled.csv', parse_dates=['timestamp'])
split_point = int(len(data) * 0.8)
train_data = data.iloc[:split_point]
test_data = data.iloc[split_point:]

feature_columns = ['memory_pct', 'roll_mean_1h', 'roll_std_1h', 'roll_mean_24h',
                   'diff_1', 'diff_6', 'hour', 'minute', 'time_of_day', 'day_of_week', 'is_weekend']

X_train = train_data[feature_columns].values
X_test = test_data[feature_columns].values
y_test = test_data['label'].values

lof_model = LocalOutlierFactor(n_neighbors=50, contamination=0.005, novelty=True)
lof_model.fit(X_train)
raw_predictions = (lof_model.predict(X_test) == -1).astype(int)
print('Raw LOF flagged:', raw_predictions.sum())

## Gap filling function
If two flagged points are within 3 positions of each other, fill the gap.

In [ ]:
def fill_gaps(predictions, max_gap=3):
    result = predictions.copy()
    detected_indices = np.where(result == 1)[0]
    for i in range(len(detected_indices) - 1):
        gap = detected_indices[i+1] - detected_indices[i]
        if 1 < gap <= max_gap + 1:
            result[detected_indices[i]+1 : detected_indices[i+1]] = 1
    return result

smoothed_predictions = fill_gaps(raw_predictions, max_gap=3)
print('After gap filling:', smoothed_predictions.sum())

## Plot before and after

In [ ]:
timestamps = test_data['timestamp'].values
memory = test_data['memory_pct'].values

plt.figure(figsize=(14, 4))
plt.plot(timestamps, memory, color='lightblue')
plt.scatter(timestamps[y_test == 1], memory[y_test == 1], color='red', marker='x', label='True anomaly')
plt.scatter(timestamps[raw_predictions == 1], memory[raw_predictions == 1], color='purple', s=10, label='Raw LOF')
plt.title('Raw LOF (before smoothing)')
plt.ylabel('Memory %')
plt.legend()
plt.show()

plt.figure(figsize=(14, 4))
plt.plot(timestamps, memory, color='lightblue')
plt.scatter(timestamps[y_test == 1], memory[y_test == 1], color='red', marker='x', label='True anomaly')
plt.scatter(timestamps[smoothed_predictions == 1], memory[smoothed_predictions == 1], color='purple', s=10, label='Smoothed LOF')
plt.title('LOF with gap filling')
plt.ylabel('Memory %')
plt.legend()
plt.show()

## Evaluation

In [ ]:
print('=== Raw LOF ===')
print(classification_report(y_test, raw_predictions, target_names=['Normal','Anomaly'], zero_division=0))
print(confusion_matrix(y_test, raw_predictions))

print()
print('=== LOF + gap filling ===')
print(classification_report(y_test, smoothed_predictions, target_names=['Normal','Anomaly'], zero_division=0))
print(confusion_matrix(y_test, smoothed_predictions))